In [26]:
import pandas as pd
df = pd.read_csv('final_report.csv')
print(df.columns)
out_of_spec_df = df[df["Remarks"] == "Out of spec"]
out_of_spec_df

Index(['Point# (XPN)', 'XPN X', 'XPN Y', 'XPN Z', 'HTol', 'LTol', 'XPR X',
       'XPR Y', 'XPR Z', 'Deviation', 'Error', 'Correction_Magnitude',
       'Remarks'],
      dtype='object')


,Point# (XPN),XPN X,XPN Y,XPN Z,HTol,LTol,XPR X,XPR Y,XPR Z,Deviation,Error,Correction_Magnitude,Remarks
0,1.0,24.461,16.317,209.0,0.08,0.0,24.499,16.247,209.0,0.08,0.0,0.843,Out of spec
8,9.0,16.919,12.625,209.0,0.08,0.0,16.953,12.552,209.0,0.08,0.0,0.821,Out of spec
12,13.0,13.564,10.068,209.0,0.08,0.0,13.616,10.007,209.0,0.08,0.0,0.785,Out of spec
13,14.0,12.752,9.394,209.0,0.08,0.0,12.797,9.328,209.0,0.08,0.0,0.840,Out of spec
15,16.0,11.156,8.014,209.0,0.08,0.0,11.205,7.951,209.0,0.08,0.0,0.814,Out of spec
...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,155.0,24.748,16.711,209.0,0.08,0.0,24.819,16.675,209.0,0.08,0.0,0.932,Out of spec
155,156.0,24.734,16.627,209.0,0.08,0.0,24.791,16.571,209.0,0.08,0.0,0.903,Out of spec
157,158.0,24.660,16.473,209.0,0.08,0.0,24.709,16.410,209.0,0.08,0.0,0.845,Out of spec
159,160.0,24.537,16.357,209.0,0.08,0.0,24.597,16.304,209.0,0.08,0.0,0.845,Out of spec


In [31]:
import numpy as np
import pandas as pd

def correct_deviation_batch(xpn_coords, xpr_coords, htol_array):
    """
    Vectorized correction for all points.
    Moves raw (XPR) points toward nominal (XPN) until within tolerance.
    """
    # Compute initial deviations
    deltas = xpn_coords - xpr_coords
    deviations = np.linalg.norm(deltas, axis=1)

    # Mask: points already within tolerance
    within_tol = deviations <= htol_array

    # Normalize direction vectors safely (avoid division by zero)
    direction_vectors = np.zeros_like(deltas)
    nonzero_mask = deviations > 0
    direction_vectors[nonzero_mask] = deltas[nonzero_mask] / deviations[nonzero_mask, None]

    # Distances to move = deviation - tolerance (only for out-of-spec points)
    correction_distances = np.maximum(deviations - htol_array, 0)

    # Apply correction
    corrected_xpr = xpr_coords + direction_vectors * correction_distances[:, None]

    # Final deviation after correction
    final_deviation = np.linalg.norm(xpn_coords - corrected_xpr, axis=1)

    # Actual correction magnitudes
    correction_magnitude = np.linalg.norm(corrected_xpr - xpr_coords, axis=1)

    # Remarks
    eps = 1e-6  # small margin for floating point errors
    remarks = np.where(final_deviation <= htol_array + eps,
                   "Within tolerance", "Out of spec")


    # remarks = np.where(final_deviation <= htol_array, "Within tolerance", "Out of spec")

    return corrected_xpr, final_deviation, correction_magnitude, remarks


# === Usage with your dataframe ===
xpn_coords = df[['XPN X', 'XPN Y', 'XPN Z']].to_numpy()
xpr_coords = df[['XPR X', 'XPR Y', 'XPR Z']].to_numpy()
htol_array = df['HTol'].to_numpy()

corrected_xpr, final_deviation, correction_magnitude, remarks = correct_deviation_batch(
    xpn_coords, xpr_coords, htol_array
)

# Update dataframe
corrected_df = df.copy()
corrected_df[['XPR X', 'XPR Y', 'XPR Z']] = corrected_xpr
corrected_df['Deviation'] = final_deviation
corrected_df['Correction_Magnitude'] = correction_magnitude
corrected_df['Remarks'] = remarks

corrected_df = corrected_df.round(3)
corrected_df

# # Out of spec points after correction
out_of_spec_df = corrected_df[corrected_df["Remarks"] == "Out of spec"]

out_of_spec_df


,Point# (XPN),XPN X,XPN Y,XPN Z,HTol,LTol,XPR X,XPR Y,XPR Z,Deviation,Error,Correction_Magnitude,Remarks
